In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:

!pip install -q -U langchain langchain-groq langgraph tavily-python


In [3]:
import langchain
import langchain_groq
import langgraph
import tavily

print("All libraries installed successfully!")

All libraries installed successfully!


In [4]:

from kaggle_secrets import UserSecretsClient
import os

# Access Kaggle Secrets
secrets = UserSecretsClient()

# Retrieve API keys
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
TAVILY_API_KEY = secrets.get_secret("TAVILY_API_KEY")

# Configure environment variables
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

# Verify configuration without displaying keys
print("Groq API Key: Configured")
print("Tavily API Key: Configured")
print("Environment setup completed successfully!")


Groq API Key: Configured
Tavily API Key: Configured
Environment setup completed successfully!


# SECTION 1: DATABASE CREATION
Creating the SQLite database for the Lab Test Booking Assistant.

In [5]:

import sqlite3

# Create database
DB_PATH = "/kaggle/working/lab_test_booking.db"

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

# 1. Tests table
conn.execute("""
CREATE TABLE IF NOT EXISTS tests (
    test_id INTEGER PRIMARY KEY AUTOINCREMENT,
    test_name TEXT UNIQUE NOT NULL,
    category TEXT NOT NULL,
    description TEXT,
    price REAL NOT NULL CHECK(price >= 0)
)
""")

# 2. Slots table
conn.execute("""
CREATE TABLE IF NOT EXISTS slots (
    slot_id INTEGER PRIMARY KEY AUTOINCREMENT,
    test_id INTEGER NOT NULL,
    slot_datetime TEXT NOT NULL,
    capacity INTEGER NOT NULL DEFAULT 1 CHECK(capacity > 0),
    booked_count INTEGER NOT NULL DEFAULT 0,
    FOREIGN KEY (test_id) REFERENCES tests(test_id),
    CHECK(booked_count >= 0 AND booked_count <= capacity),
    UNIQUE(test_id, slot_datetime)
)
""")

# 3. Bookings table
conn.execute("""
CREATE TABLE IF NOT EXISTS bookings (
    booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT NOT NULL,
    test_id INTEGER NOT NULL,
    slot_id INTEGER NOT NULL,
    booking_status TEXT NOT NULL DEFAULT 'booked'
        CHECK(booking_status IN ('booked', 'cancelled')),
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (test_id) REFERENCES tests(test_id),
    FOREIGN KEY (slot_id) REFERENCES slots(slot_id)
)
""")

# Prevent duplicate active bookings for the same test by a user
conn.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS unique_active_booking
ON bookings(user_id, test_id)
WHERE booking_status = 'booked'
""")

conn.commit()

print("SQLite database created successfully!")
print("Database location:", DB_PATH)
print("Tables created: tests, slots, bookings")

conn.close()


SQLite database created successfully!
Database location: /kaggle/working/lab_test_booking.db
Tables created: tests, slots, bookings


# SECTION 2: DATABASE SEEDING
Inserting diagnostic tests and appointment slots into SQLite.

In [6]:

import sqlite3

DB_PATH = "/kaggle/working/lab_test_booking.db"

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

# 1. Diagnostic tests (fictional demonstration data)
tests_data = [
    ("Complete Blood Count (CBC)", "Hematology", "Measures blood cells and hemoglobin.", 450),
    ("Hemoglobin (HB)", "Hematology", "Measures hemoglobin levels in blood.", 200),
    ("Blood Group Test", "Hematology", "Identifies ABO and Rh blood groups.", 250),
    ("Blood Glucose (Fasting)", "Biochemistry", "Measures fasting blood glucose.", 150),
    ("Lipid Profile", "Biochemistry", "Measures cholesterol and triglycerides.", 800),
    ("Liver Function Test (LFT)", "Biochemistry", "Measures markers related to liver function.", 700),
    ("Kidney Function Test (KFT)", "Biochemistry", "Measures markers related to kidney function.", 650),
    ("Electrocardiogram (ECG)", "Cardiology", "Records the electrical activity of the heart.", 500)
]

conn.executemany("""
INSERT OR IGNORE INTO tests
(test_name, category, description, price)
VALUES (?, ?, ?, ?)
""", tests_data)

# 2. Appointment slots
slot_times = [
    "2026-12-01 09:00:00",
    "2026-12-01 10:00:00",
    "2026-12-01 11:00:00"
]

# Retrieve test IDs from database
test_rows = conn.execute("""
SELECT test_id, test_name
FROM tests
""").fetchall()

for test_id, test_name in test_rows:

    # ECG is intentionally fully booked
    if test_name == "Electrocardiogram (ECG)":
        booked_count = 1
    else:
        booked_count = 0

    for slot_time in slot_times:
        conn.execute("""
        INSERT OR IGNORE INTO slots
        (test_id, slot_datetime, capacity, booked_count)
        VALUES (?, ?, ?, ?)
        """, (test_id, slot_time, 1, booked_count))

conn.commit()

print("Diagnostic tests and slots inserted successfully!")

# 3. Display seeded tests
print("\n===== DIAGNOSTIC TESTS =====")

for row in conn.execute("""
SELECT test_id, test_name, category, price
FROM tests
ORDER BY test_id
"""):
    print(row)

# 4. Display slot counts
print("\n===== SLOT SUMMARY =====")

for row in conn.execute("""
SELECT t.test_name,
       COUNT(s.slot_id) AS total_slots,
       SUM(CASE WHEN s.booked_count < s.capacity THEN 1 ELSE 0 END)
           AS free_slots
FROM tests t
JOIN slots s ON t.test_id = s.test_id
GROUP BY t.test_id, t.test_name
ORDER BY t.test_id
"""):
    print(row)

conn.close()


Diagnostic tests and slots inserted successfully!

===== DIAGNOSTIC TESTS =====
(1, 'Complete Blood Count (CBC)', 'Hematology', 450.0)
(2, 'Hemoglobin (HB)', 'Hematology', 200.0)
(3, 'Blood Group Test', 'Hematology', 250.0)
(4, 'Blood Glucose (Fasting)', 'Biochemistry', 150.0)
(5, 'Lipid Profile', 'Biochemistry', 800.0)
(6, 'Liver Function Test (LFT)', 'Biochemistry', 700.0)
(7, 'Kidney Function Test (KFT)', 'Biochemistry', 650.0)
(8, 'Electrocardiogram (ECG)', 'Cardiology', 500.0)

===== SLOT SUMMARY =====
('Complete Blood Count (CBC)', 3, 3)
('Hemoglobin (HB)', 3, 3)
('Blood Group Test', 3, 3)
('Blood Glucose (Fasting)', 3, 3)
('Lipid Profile', 3, 3)
('Liver Function Test (LFT)', 3, 3)
('Kidney Function Test (KFT)', 3, 3)
('Electrocardiogram (ECG)', 3, 0)


In [7]:

import sqlite3

conn = sqlite3.connect(DB_PATH)

# Count records
test_count = conn.execute(
    "SELECT COUNT(*) FROM tests"
).fetchone()[0]

slot_count = conn.execute(
    "SELECT COUNT(*) FROM slots"
).fetchone()[0]

# Check duplicate test names
duplicate_tests = conn.execute("""
SELECT test_name, COUNT(*)
FROM tests
GROUP BY test_name
HAVING COUNT(*) > 1
""").fetchall()

# Check duplicate slots
duplicate_slots = conn.execute("""
SELECT test_id, slot_datetime, COUNT(*)
FROM slots
GROUP BY test_id, slot_datetime
HAVING COUNT(*) > 1
""").fetchall()

# Check fully booked test
fully_booked = conn.execute("""
SELECT t.test_name
FROM tests t
JOIN slots s ON t.test_id = s.test_id
GROUP BY t.test_id
HAVING SUM(s.capacity - s.booked_count) = 0
""").fetchall()

print("===== DATABASE VERIFICATION =====")
print("Total tests:", test_count)
print("Total slots:", slot_count)
print("Duplicate tests:", duplicate_tests)
print("Duplicate slots:", duplicate_slots)
print("Tests with no free slots:", fully_booked)

print("\nDatabase verification:",
      "PASSED" if test_count == 8
      and slot_count == 24
      and not duplicate_tests
      and not duplicate_slots
      and len(fully_booked) == 1
      else "FAILED")

conn.close()


===== DATABASE VERIFICATION =====
Total tests: 8
Total slots: 24
Duplicate tests: []
Duplicate slots: []
Tests with no free slots: [('Electrocardiogram (ECG)',)]

Database verification: PASSED


In [8]:

import sqlite3
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

DB_PATH = "/kaggle/working/lab_test_booking.db"

# User identity is supplied by the application, not the AI model.
@dataclass
class UserContext:
    user_id: str


# TOOL 1: List all diagnostic tests
@tool
def list_tests(category: str = "") -> str:
    """
    List diagnostic tests with their categories, descriptions and prices.
    Optionally filter by category.
    Prices must always be retrieved from the database.
    """
    conn = sqlite3.connect(DB_PATH)

    rows = conn.execute("""
        SELECT test_id, test_name, category, description, price
        FROM tests
        WHERE (? = '' OR LOWER(category) = LOWER(?))
        ORDER BY test_id
    """, (category, category)).fetchall()

    conn.close()

    if not rows:
        return "No diagnostic tests found."

    result = ["AVAILABLE DIAGNOSTIC TESTS:"]

    for row in rows:
        test_id, name, cat, description, price = row

        result.append(
            f"ID: {test_id} | Test: {name} | "
            f"Category: {cat} | Description: {description} | "
            f"Price: Rs. {price:.2f}"
        )

    return "\n".join(result)


# TOOL 2: Check available appointment slots
@tool
def check_slots(test_name: str) -> str:
    """
    Check available appointment slots for a diagnostic test.
    Returns only availability retrieved from SQLite.
    """
    conn = sqlite3.connect(DB_PATH)

    rows = conn.execute("""
        SELECT s.slot_id, s.slot_datetime,
               s.capacity - s.booked_count AS available
        FROM slots s
        JOIN tests t ON s.test_id = t.test_id
        WHERE LOWER(t.test_name) = LOWER(?)
        ORDER BY s.slot_datetime
    """, (test_name,)).fetchall()

    conn.close()

    if not rows:
        return f"No slots found for test: {test_name}"

    free_slots = [
        row for row in rows if row[2] > 0
    ]

    if not free_slots:
        return f"No free slots are currently available for {test_name}."

    result = [f"AVAILABLE SLOTS FOR {test_name}:"]

    for slot_id, slot_datetime, available in free_slots:
        result.append(
            f"Slot ID: {slot_id} | "
            f"Date and Time: {slot_datetime} | "
            f"Available Capacity: {available}"
        )

    return "\n".join(result)


# TOOL 3: View bookings of the authenticated user
@tool
def my_bookings(runtime: ToolRuntime[UserContext]) -> str:
    """
    Display bookings belonging only to the authenticated user.
    User identity comes from runtime context, never from model arguments.
    """
    user_id = runtime.context.user_id

    conn = sqlite3.connect(DB_PATH)

    rows = conn.execute("""
        SELECT b.booking_id, t.test_name,
               s.slot_datetime, b.booking_status
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.user_id = ?
        ORDER BY b.booking_id DESC
    """, (user_id,)).fetchall()

    conn.close()

    if not rows:
        return "You have no bookings."

    result = ["YOUR BOOKINGS:"]

    for booking_id, test_name, slot_datetime, status in rows:
        result.append(
            f"Booking ID: {booking_id} | "
            f"Test: {test_name} | "
            f"Date: {slot_datetime} | "
            f"Status: {status}"
        )

    return "\n".join(result)


print("Three read tools created successfully!")
print("1. list_tests")
print("2. check_slots")
print("3. my_bookings")


Three read tools created successfully!
1. list_tests
2. check_slots
3. my_bookings


In [9]:

print(list_tests.invoke({}))


AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Category: Biochemistry | Description: Measures markers related to kidney function. | Price: Rs. 650.00
ID: 8 | Test: Electrocardiogram (ECG) 

In [10]:

# Check a test with available slots
print(check_slots.invoke({
    "test_name": "Complete Blood Count (CBC)"
}))

print("\n===== FULLY BOOKED TEST =====")

# Check the test with no free slots
print(check_slots.invoke({
    "test_name": "Electrocardiogram (ECG)"
}))


AVAILABLE SLOTS FOR Complete Blood Count (CBC):
Slot ID: 7 | Date and Time: 2026-12-01 09:00:00 | Available Capacity: 1
Slot ID: 8 | Date and Time: 2026-12-01 10:00:00 | Available Capacity: 1
Slot ID: 9 | Date and Time: 2026-12-01 11:00:00 | Available Capacity: 1

===== FULLY BOOKED TEST =====
No free slots are currently available for Electrocardiogram (ECG).


In [11]:

print("===== MODEL-VISIBLE TOOL SCHEMA =====")

# Get the schema exposed to the AI model
schema = my_bookings.tool_call_schema.model_json_schema()

print(schema)

# Verify that user_id is hidden from the model
properties = schema.get("properties", {})

print("\n===== SECURITY VERIFICATION =====")
print("Model-visible arguments:", list(properties.keys()))
print("Does model control user_id?", "user_id" in properties)

if "user_id" not in properties:
    print("\nSECURITY CHECK PASSED!")
else:
    print("\nSECURITY CHECK FAILED!")


===== MODEL-VISIBLE TOOL SCHEMA =====
{'description': 'Display bookings belonging only to the authenticated user.\nUser identity comes from runtime context, never from model arguments.', 'properties': {}, 'title': 'my_bookings', 'type': 'object'}

===== SECURITY VERIFICATION =====
Model-visible arguments: []
Does model control user_id? False

SECURITY CHECK PASSED!


In [12]:
from langchain.tools import tool, ToolRuntime
from dataclasses import dataclass
import sqlite3

DB_PATH = "/kaggle/working/lab_test_booking.db"

@dataclass
class UserContext:
    user_id: str


@tool
def book_test(
    test_name: str,
    slot_id: int,
    runtime: ToolRuntime[UserContext]
) -> str:
    """
    Book a diagnostic test for the authenticated user.
    Requires a valid test and available slot.
    User identity comes from runtime context.
    """
    user_id = runtime.context.user_id
    conn = sqlite3.connect(DB_PATH)

    try:
        conn.execute("BEGIN IMMEDIATE")

        test = conn.execute(
            "SELECT test_id, price FROM tests WHERE LOWER(test_name) = LOWER(?)",
            (test_name,)
        ).fetchone()

        if not test:
            conn.rollback()
            return "Booking failed: Test not found."

        test_id, price = test

        slot = conn.execute(
            """
            SELECT slot_id, slot_datetime, capacity, booked_count
            FROM slots
            WHERE slot_id = ? AND test_id = ?
            """,
            (slot_id, test_id)
        ).fetchone()

        if not slot:
            conn.rollback()
            return "Booking failed: Invalid slot for this test."

        _, slot_datetime, capacity, booked_count = slot

        if booked_count >= capacity:
            conn.rollback()
            return "Booking failed: This slot is full."

        duplicate = conn.execute(
            """
            SELECT booking_id FROM bookings
            WHERE user_id = ? AND test_id = ?
              AND booking_status = 'booked'
            """,
            (user_id, test_id)
        ).fetchone()

        if duplicate:
            conn.rollback()
            return "Booking failed: You already have an active booking for this test."

        conn.execute(
            """
            INSERT INTO bookings (user_id, test_id, slot_id)
            VALUES (?, ?, ?)
            """,
            (user_id, test_id, slot_id)
        )

        conn.execute(
            """
            UPDATE slots
            SET booked_count = booked_count + 1
            WHERE slot_id = ?
            """,
            (slot_id,)
        )

        booking = conn.execute(
            """
            SELECT b.booking_id, t.test_name, s.slot_datetime,
                   b.booking_status, t.price
            FROM bookings b
            JOIN tests t ON b.test_id = t.test_id
            JOIN slots s ON b.slot_id = s.slot_id
            WHERE b.user_id = ? AND b.test_id = ?
              AND b.booking_status = 'booked'
            ORDER BY b.booking_id DESC
            LIMIT 1
            """,
            (user_id, test_id)
        ).fetchone()

        conn.commit()

        booking_id, name, date_time, status, db_price = booking

        return (
            "BOOKING CONFIRMED!\n"
            f"Booking ID: {booking_id}\n"
            f"User ID: {user_id}\n"
            f"Test: {name}\n"
            f"Date and Time: {date_time}\n"
            f"Price: Rs. {db_price:.2f}\n"
            f"Status: {status}"
        )

    except sqlite3.IntegrityError:
        conn.rollback()
        return "Booking failed: A duplicate or invalid booking was detected."
    finally:
        conn.close()


@tool
def cancel_booking(
    booking_id: int,
    runtime: ToolRuntime[UserContext]
) -> str:
    """
    Cancel a booking belonging only to the authenticated user.
    User identity comes from runtime context.
    """
    user_id = runtime.context.user_id
    conn = sqlite3.connect(DB_PATH)

    try:
        conn.execute("BEGIN IMMEDIATE")

        booking = conn.execute(
            """
            SELECT booking_id, slot_id
            FROM bookings
            WHERE booking_id = ?
              AND user_id = ?
              AND booking_status = 'booked'
            """,
            (booking_id, user_id)
        ).fetchone()

        if not booking:
            conn.rollback()
            return "Cancellation failed: Active booking not found for your account."

        _, slot_id = booking

        conn.execute(
            """
            UPDATE bookings
            SET booking_status = 'cancelled'
            WHERE booking_id = ? AND user_id = ?
            """,
            (booking_id, user_id)
        )

        conn.execute(
            """
            UPDATE slots
            SET booked_count = booked_count - 1
            WHERE slot_id = ? AND booked_count > 0
            """,
            (slot_id,)
        )

        conn.commit()
        return f"Booking {booking_id} cancelled successfully."

    except sqlite3.Error as error:
        conn.rollback()
        return f"Cancellation failed: {error}"
    finally:
        conn.close()


print("Write tools created successfully!")
print("1. book_test")
print("2. cancel_booking")

Write tools created successfully!
1. book_test
2. cancel_booking


In [13]:
print("===== BOOK_TEST TOOL SCHEMA =====")
book_schema = book_test.tool_call_schema.model_json_schema()
print(book_schema)

print("\n===== CANCEL_BOOKING TOOL SCHEMA =====")
cancel_schema = cancel_booking.tool_call_schema.model_json_schema()
print(cancel_schema)

print("\n===== SECURITY VERIFICATION =====")

book_args = book_schema.get("properties", {})
cancel_args = cancel_schema.get("properties", {})

print("book_test arguments:", list(book_args.keys()))
print("cancel_booking arguments:", list(cancel_args.keys()))

book_secure = "user_id" not in book_args
cancel_secure = "user_id" not in cancel_args

print("\nbook_test hides user_id:", book_secure)
print("cancel_booking hides user_id:", cancel_secure)

if book_secure and cancel_secure:
    print("\nSECURITY CHECK PASSED!")
else:
    print("\nSECURITY CHECK FAILED!")

===== BOOK_TEST TOOL SCHEMA =====
{'description': 'Book a diagnostic test for the authenticated user.\nRequires a valid test and available slot.\nUser identity comes from runtime context.', 'properties': {'test_name': {'title': 'Test Name', 'type': 'string'}, 'slot_id': {'title': 'Slot Id', 'type': 'integer'}}, 'required': ['test_name', 'slot_id'], 'title': 'book_test', 'type': 'object'}

===== CANCEL_BOOKING TOOL SCHEMA =====
{'description': 'Cancel a booking belonging only to the authenticated user.\nUser identity comes from runtime context.', 'properties': {'booking_id': {'title': 'Booking Id', 'type': 'integer'}}, 'required': ['booking_id'], 'title': 'cancel_booking', 'type': 'object'}

===== SECURITY VERIFICATION =====
book_test arguments: ['test_name', 'slot_id']
cancel_booking arguments: ['booking_id']

book_test hides user_id: True
cancel_booking hides user_id: True

SECURITY CHECK PASSED!


In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

response = llm.invoke(
    "Reply with: Lab Booking Assistant model connected!"
)

print(response.content)

Lab Booking Assistant model connected!


In [15]:
tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking
]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "Show me the available diagnostic tests."
)

print("AI Response:")
print(response.content)

print("\nTool calls requested by model:")
for call in response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

AI Response:


Tool calls requested by model:
Tool: list_tests
Arguments: {'category': ''}


In [16]:
!pip install -q -U langchain-tavily

In [17]:
from langchain_tavily import TavilySearch

print("Tavily LangChain integration installed successfully!")

Tavily LangChain integration installed successfully!


In [18]:
from langchain_tavily import TavilySearch
from langchain.tools import tool

tavily_search = TavilySearch(
    max_results=3,
    topic="general"
)

@tool
def search_health_information(query: str) -> str:
    """
    Search general health information about diagnostic tests,
    fasting requirements and what laboratory tests measure.

    Never use web search for laboratory prices, appointment slots,
    booking availability or medical diagnosis.
    """
    return str(tavily_search.invoke({"query": query}))

print("Health information search tool created successfully!")

Health information search tool created successfully!


In [19]:
result = search_health_information.invoke({
    "query": "What does a Complete Blood Count (CBC) test measure?"
})

print("===== HEALTH INFORMATION SEARCH RESULTS =====")
print(result)

===== HEALTH INFORMATION SEARCH RESULTS =====
{'query': 'What does a Complete Blood Count (CBC) test measure?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://together.stjude.org/en-us/treatment-tests-procedures/tests/complete-blood-count-tests.html', 'title': 'Complete Blood Count (CBC) Test - Together by St. Jude™', 'content': 'A complete blood count (CBC) test measures the number and types of cells in the blood, including red blood cells, white blood cells, platelets, and other parts of the blood.\n A CBC test can help find infections, blood disorders, signs of cancer, and other health problems.\n A CBC with differential gives more details about white blood cells and infection risk.\n A CBC test can monitor your child’s health and response to treatment. [...] ## What is a complete blood count test?\n\nA complete blood count (CBC) is a lab test that measures the types and numbers of blood cells, such as:\n\n Red blood cells\n White blood cells

In [20]:
import inspect
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

print("create_agent:", inspect.signature(create_agent))
print("\nHumanInTheLoopMiddleware:")
print(inspect.signature(HumanInTheLoopMiddleware))

create_agent: (model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None, transformers: 'Sequence[TransformerFactory] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]'

HumanInTheLoopMiddleware:
(interrupt_on: 'dict[str, bool | InterruptOnConfig]', *, description_prefix: 'str' = 'Tool execution req

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# All six tools
all_tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking,
    search_health_information
]

# Human approval for write operations only
approval_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "book_test": True,
        "cancel_booking": True
    }
)

# Conversation memory
memory = InMemorySaver()

# Agent instructions
system_prompt = """
You are a helpful Lab Test Booking Assistant.

RULES:

1. Use list_tests to retrieve test names, categories and prices.
2. Use check_slots to retrieve appointment availability.
3. Use my_bookings to display the authenticated user's bookings.
4. Use book_test to make bookings, but never bypass human approval.
5. Use cancel_booking to cancel bookings, but never bypass human approval.
6. Use search_health_information only for general health information.
7. Never use web search for prices, laboratory availability or bookings.
8. Never invent prices, slots or booking confirmations.
9. Never diagnose diseases.
10. Never recommend or select a diagnostic test based on symptoms.
11. For symptom-related questions, advise the user to consult a qualified doctor.
12. Before booking, clearly explain the selected test, price and slot.
13. Never claim a booking or cancellation succeeded until the corresponding
    database tool confirms it.
14. Be polite, clear and concise.
"""

# Create agent
agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=system_prompt,
    context_schema=UserContext,
    checkpointer=memory,
    middleware=[approval_middleware]
)

print("Lab Test Booking AI Agent created successfully!")
print("Tools connected:", len(all_tools))
print("Conversation memory: Enabled")
print("Booking approval: Enabled")
print("Cancellation approval: Enabled")

Lab Test Booking AI Agent created successfully!
Tools connected: 6
Conversation memory: Enabled
Booking approval: Enabled
Cancellation approval: Enabled


In [22]:
from langchain_core.messages import HumanMessage

test_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Show me all available diagnostic tests and their prices."
            )
        ]
    },
    config={
        "configurable": {
            "thread_id": "test-user-001"
        }
    },
    context=UserContext(user_id="user_001")
)

print("===== AGENT RESPONSE =====")

for message in test_response["messages"]:
    print("\n", message.type.upper(), ":")
    print(message.content)

    if message.type == "ai" and message.tool_calls:
        for call in message.tool_calls:
            print("TOOL CALLED:", call["name"])

===== AGENT RESPONSE =====

 HUMAN :
Show me all available diagnostic tests and their prices.

 AI :

TOOL CALLED: list_tests

 TOOL :
AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Catego

In [23]:
slot_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Check available appointment slots for Complete Blood Count (CBC)."
            )
        ]
    },
    config={
        "configurable": {
            "thread_id": "test-user-001"
        }
    },
    context=UserContext(user_id="user_001")
)

print("===== SLOT CHECK RESPONSE =====")

for message in slot_response["messages"]:
    if message.type == "ai":
        print("\nAI:", message.content)

        for call in message.tool_calls:
            print("TOOL CALLED:", call["name"])

    elif message.type == "tool":
        print("\nTOOL RESULT:", message.content)

===== SLOT CHECK RESPONSE =====

AI: 
TOOL CALLED: list_tests

TOOL RESULT: AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Category: Biochemistry | Description: Measures markers related to

In [24]:
memory_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="What diagnostic test did I ask you to check appointment availability for earlier?"
            )
        ]
    },
    config={
        "configurable": {
            "thread_id": "test-user-001"
        }
    },
    context=UserContext(user_id="user_001")
)

print("===== MEMORY TEST =====")

print(memory_response["messages"][-1].content)

===== MEMORY TEST =====
You asked to check the appointment availability for the **Complete Blood Count (CBC)** test.


In [25]:
def show_booking_database():
    conn = sqlite3.connect(DB_PATH)

    print("===== CURRENT BOOKINGS =====")
    rows = conn.execute("""
        SELECT booking_id, user_id, test_id, slot_id, booking_status
        FROM bookings
        ORDER BY booking_id
    """).fetchall()

    print(rows if rows else "No bookings yet.")

    print("\n===== SLOT COUNTS =====")
    slots = conn.execute("""
        SELECT slot_id, capacity, booked_count
        FROM slots
        ORDER BY slot_id
    """).fetchall()

    for row in slots:
        print(row)

    conn.close()


show_booking_database()

===== CURRENT BOOKINGS =====
No bookings yet.

===== SLOT COUNTS =====
(1, 1, 0)
(2, 1, 0)
(3, 1, 0)
(4, 1, 0)
(5, 1, 0)
(6, 1, 0)
(7, 1, 0)
(8, 1, 0)
(9, 1, 0)
(10, 1, 1)
(11, 1, 1)
(12, 1, 1)
(13, 1, 0)
(14, 1, 0)
(15, 1, 0)
(16, 1, 0)
(17, 1, 0)
(18, 1, 0)
(19, 1, 0)
(20, 1, 0)
(21, 1, 0)
(22, 1, 0)
(23, 1, 0)
(24, 1, 0)


In [26]:
from langchain_core.messages import HumanMessage

approval_thread = "approval-test-user-001"

booking_request = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "I want to book Complete Blood Count (CBC) "
                    "for appointment slot ID 7. Please proceed."
                )
            )
        ]
    },
    config={"configurable": {"thread_id": approval_thread}},
    context=UserContext(user_id="user_001")
)

print("===== AGENT RESPONSE =====")

for message in booking_request["messages"]:
    print("\nMessage type:", type(message).__name__)
    print("Content:", message.content)

print("\n===== APPROVAL INTERRUPT =====")
print(booking_request.get("__interrupt__", "No interrupt found"))

===== AGENT RESPONSE =====

Message type: HumanMessage
Content: I want to book Complete Blood Count (CBC) for appointment slot ID 7. Please proceed.

Message type: AIMessage
Content: 

Message type: ToolMessage
Content: AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to 

In [27]:
from langchain_core.messages import HumanMessage

confirmed_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Yes, I confirm. Please book the CBC test in slot ID 7."
            )
        ]
    },
    config={"configurable": {"thread_id": approval_thread}},
    context=UserContext(user_id="user_001")
)

print("===== AGENT RESPONSE =====")

for message in confirmed_response["messages"]:
    print("\nMessage type:", type(message).__name__)
    print("Content:", message.content)

print("\n===== APPROVAL INTERRUPT =====")
print(confirmed_response.get("__interrupt__", "No interrupt found"))

===== AGENT RESPONSE =====

Message type: HumanMessage
Content: I want to book Complete Blood Count (CBC) for appointment slot ID 7. Please proceed.

Message type: AIMessage
Content: 

Message type: ToolMessage
Content: AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to 

In [28]:
from langgraph.types import Command

approved_response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config={"configurable": {"thread_id": approval_thread}},
    context=UserContext(user_id="user_001")
)

print("===== APPROVED BOOKING RESPONSE =====")

for message in approved_response["messages"]:
    print("\nMessage type:", type(message).__name__)
    print("Content:", message.content)

print("\n===== DATABASE AFTER APPROVAL =====")
show_booking_database()

===== APPROVED BOOKING RESPONSE =====

Message type: HumanMessage
Content: I want to book Complete Blood Count (CBC) for appointment slot ID 7. Please proceed.

Message type: AIMessage
Content: 

Message type: ToolMessage
Content: AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers 

In [29]:
rejection_thread = "rejection-test-user-001"

rejection_request = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "I want to book Complete Blood Count (CBC) "
                    "for appointment slot ID 8."
                )
            )
        ]
    },
    config={"configurable": {"thread_id": rejection_thread}},
    context=UserContext(user_id="user_001")
)

for message in rejection_request["messages"]:
    print("\n", type(message).__name__, ":", message.content)


 HumanMessage : I want to book Complete Blood Count (CBC) for appointment slot ID 8.

 AIMessage : 

 ToolMessage : AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Category: Biochemistry |

In [30]:
rejection_confirmed = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Yes, I confirm. Proceed with the booking."
            )
        ]
    },
    config={"configurable": {"thread_id": rejection_thread}},
    context=UserContext(user_id="user_001")
)

print("===== REJECTION TEST APPROVAL REQUEST =====")

for message in rejection_confirmed["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== INTERRUPT =====")
print(rejection_confirmed.get("__interrupt__", "No interrupt found"))

===== REJECTION TEST APPROVAL REQUEST =====

 HumanMessage : I want to book Complete Blood Count (CBC) for appointment slot ID 8.

 AIMessage : 

 ToolMessage : AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Fu

In [31]:
rejected_response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "Booking rejected for testing purposes."
                }
            ]
        }
    ),
    config={"configurable": {"thread_id": rejection_thread}},
    context=UserContext(user_id="user_001")
)

print("===== REJECTION RESULT =====")

for message in rejected_response["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== DATABASE AFTER REJECTION =====")
show_booking_database()

===== REJECTION RESULT =====

 HumanMessage : I want to book Complete Blood Count (CBC) for appointment slot ID 8.

 AIMessage : 

 ToolMessage : AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KF

In [32]:
# Keep track of the simulated failure
failure_state = {"failed": False}

# Save the original tool function
original_check_slots_func = check_slots.func

def unreliable_check_slots(test_name: str) -> str:
    if not failure_state["failed"]:
        failure_state["failed"] = True
        print("SIMULATED FAILURE: check_slots failed temporarily!")
        raise RuntimeError("Temporary database connection error")

    print("RETRY: check_slots is running again...")
    return original_check_slots_func(test_name)

# Create a new tool using the simulated function
from langchain_core.tools import StructuredTool

check_slots_retry_tool = StructuredTool.from_function(
    func=unreliable_check_slots,
    name="check_slots",
    description=check_slots.description,
    args_schema=check_slots.args_schema
)

# Configure automatic retry
check_slots_retry_tool = check_slots_retry_tool.with_retry(
    stop_after_attempt=2,
    retry_if_exception_type=(RuntimeError,),
    wait_exponential_jitter=False
)

print("Retry-enabled check_slots tool created!")
print("Maximum attempts: 2")

Retry-enabled check_slots tool created!
Maximum attempts: 2


In [33]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

@tool("check_slots")
def check_slots_with_retry(test_name: str) -> str:
    """Check available appointment slots with automatic retry."""
    return check_slots_retry_tool.invoke({"test_name": test_name})


# Replace only check_slots; keep all other tools unchanged
retry_tools = [
    list_tests,
    check_slots_with_retry,
    my_bookings,
    book_test,
    cancel_booking,
    search_health_information
]

retry_agent = create_agent(
    model=llm,
    tools=retry_tools,
    system_prompt=system_prompt,
    context_schema=UserContext,
    checkpointer=InMemorySaver(),
    middleware=[approval_middleware]
)

print("Reliability test agent created successfully!")
print("Tools connected:", len(retry_tools))
print("check_slots: Automatic retry enabled")

Reliability test agent created successfully!
Tools connected: 6
check_slots: Automatic retry enabled


In [34]:
failure_state["failed"] = False

retry_response = retry_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Check available appointment slots for Complete Blood Count (CBC)."
            )
        ]
    },
    config={"configurable": {"thread_id": "retry-test-user-001"}},
    context=UserContext(user_id="user_002")
)

print("\n===== RETRY AGENT FINAL RESPONSE =====")
print(retry_response["messages"][-1].content)

print("\n===== FAILURE SIMULATION STATUS =====")
print("First attempt failed:", failure_state["failed"])
print("Recovery completed: Check the successful slot result above.")

SIMULATED FAILURE: check_slots failed temporarily!
RETRY: check_slots is running again...

===== RETRY AGENT FINAL RESPONSE =====
Here are the upcoming appointment slots that are currently open for the **Complete Blood Count (CBC)** test:

| Slot ID | Date & Time (local) | Available Capacity |
|--------|----------------------|--------------------|
| 8 | 2026‑12‑01 10:00 | 1 |
| 9 | 2026‑12‑01 11:00 | 1 |

Let me know which slot you’d like to reserve, and I’ll walk you through the booking process (including the test price) before confirming.

===== FAILURE SIMULATION STATUS =====
First attempt failed: True
Recovery completed: Check the successful slot result above.


In [35]:
import re

def redact_email(text):
    email_pattern = (
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b'
    )

    return re.sub(
        email_pattern,
        "[REDACTED_EMAIL]",
        text
    )

print("Email redaction function created!")

Email redaction function created!


In [36]:
# Simulated user message
original_message = (
    "Hello, my email is student@example.com. "
    "Please show me my bookings."
)

# Redact email before sending to the model
safe_message = redact_email(original_message)

print("===== ORIGINAL MESSAGE =====")
print(original_message)

print("\n===== MESSAGE SENT TO AI =====")
print(safe_message)

print("\n===== PRIVACY VERIFICATION =====")
print("Original email present:", "student@example.com" in original_message)
print("Original email leaked:", "student@example.com" in safe_message)
print("Redaction successful:", "[REDACTED_EMAIL]" in safe_message)

===== ORIGINAL MESSAGE =====
Hello, my email is student@example.com. Please show me my bookings.

===== MESSAGE SENT TO AI =====
Hello, my email is [REDACTED_EMAIL]. Please show me my bookings.

===== PRIVACY VERIFICATION =====
Original email present: True
Original email leaked: False
Redaction successful: True


In [37]:
privacy_response = agent.invoke(
    {
        "messages": [
            HumanMessage(content=safe_message)
        ]
    },
    config={"configurable": {"thread_id": "privacy-test-001"}},
    context=UserContext(user_id="user_001")
)

print("===== MESSAGE RECEIVED BY AGENT =====")
print(privacy_response["messages"][0].content)

print("\n===== PRIVACY CHECK =====")
print(
    "Email exposed to agent:",
    "student@example.com" in privacy_response["messages"][0].content
)

===== MESSAGE RECEIVED BY AGENT =====
Hello, my email is [REDACTED_EMAIL]. Please show me my bookings.

===== PRIVACY CHECK =====
Email exposed to agent: False


In [38]:
user1_response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Show me my bookings.")
        ]
    },
    config={"configurable": {"thread_id": "security-user-1"}},
    context=UserContext(user_id="user_001")
)

print("===== USER 1 BOOKINGS =====")
print(user1_response["messages"][-1].content)

===== USER 1 BOOKINGS =====
Here are your current bookings:

| Booking ID | Test | Date & Time | Status |
|------------|------|-------------|--------|
| 1 | Complete Blood Count (CBC) | 2026‑12‑01 09:00 | booked |

Let me know if you’d like to add a new test, change this appointment, or cancel it. I'm happy to help!


In [39]:
user2_response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Show me my bookings.")
        ]
    },
    config={"configurable": {"thread_id": "security-user-2"}},
    context=UserContext(user_id="user_002")
)

print("===== USER 2 BOOKINGS =====")
print(user2_response["messages"][-1].content)

===== USER 2 BOOKINGS =====
Here are your current bookings:

*You have no bookings at the moment.*

If you’d like to schedule a new lab test, just let me know which test you’re interested in, and I can check the available slots for you.


In [40]:
unauthorized_cancel_thread = "security-cancel-user-2"

unauthorized_cancel_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Cancel booking ID 1. Please proceed."
            )
        ]
    },
    config={"configurable": {"thread_id": unauthorized_cancel_thread}},
    context=UserContext(user_id="user_002")
)

print("===== USER 2 CANCELLATION REQUEST =====")

for message in unauthorized_cancel_response["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== APPROVAL INTERRUPT =====")
print(unauthorized_cancel_response.get("__interrupt__", "No interrupt found"))

===== USER 2 CANCELLATION REQUEST =====

 HumanMessage : Cancel booking ID 1. Please proceed.

 AIMessage : Sure, I can help with that. Just to confirm: you’d like to cancel the booking with ID 1, correct? Please let me know if you’d like to proceed.

===== APPROVAL INTERRUPT =====
No interrupt found


In [41]:
unauthorized_cancel_result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config={"configurable": {"thread_id": unauthorized_cancel_thread}},
    context=UserContext(user_id="user_002")
)

print("===== UNAUTHORIZED CANCELLATION RESULT =====")

for message in unauthorized_cancel_result["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== DATABASE AFTER ATTEMPT =====")
show_booking_database()

===== UNAUTHORIZED CANCELLATION RESULT =====

 HumanMessage : Cancel booking ID 1. Please proceed.

 AIMessage : Sure, I can help with that. Just to confirm: you’d like to cancel the booking with ID 1, correct? Please let me know if you’d like to proceed.

===== DATABASE AFTER ATTEMPT =====
===== CURRENT BOOKINGS =====
[(1, 'user_001', 1, 7, 'booked')]

===== SLOT COUNTS =====
(1, 1, 0)
(2, 1, 0)
(3, 1, 0)
(4, 1, 0)
(5, 1, 0)
(6, 1, 0)
(7, 1, 1)
(8, 1, 0)
(9, 1, 0)
(10, 1, 1)
(11, 1, 1)
(12, 1, 1)
(13, 1, 0)
(14, 1, 0)
(15, 1, 0)
(16, 1, 0)
(17, 1, 0)
(18, 1, 0)
(19, 1, 0)
(20, 1, 0)
(21, 1, 0)
(22, 1, 0)
(23, 1, 0)
(24, 1, 0)


In [42]:
authorized_cancel_thread = "authorized-cancel-user-1"

authorized_cancel_request = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="I want to cancel my booking ID 1. Please proceed."
            )
        ]
    },
    config={"configurable": {"thread_id": authorized_cancel_thread}},
    context=UserContext(user_id="user_001")
)

print("===== AUTHORIZED CANCELLATION REQUEST =====")

for message in authorized_cancel_request["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== APPROVAL INTERRUPT =====")
print(authorized_cancel_request.get("__interrupt__", "No interrupt found"))

===== AUTHORIZED CANCELLATION REQUEST =====

 HumanMessage : I want to cancel my booking ID 1. Please proceed.

 AIMessage : 

===== APPROVAL INTERRUPT =====
[Interrupt(value={'action_requests': [{'name': 'cancel_booking', 'args': {'booking_id': 1}, 'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: {'booking_id': 1}"}], 'review_configs': [{'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='1dd60e012b841394a65f76fb5791a02d', response_schema=None)]


In [43]:
from langgraph.types import Command

authorized_cancel_result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config={"configurable": {"thread_id": authorized_cancel_thread}},
    context=UserContext(user_id="user_001")
)

print("===== AUTHORIZED CANCELLATION RESULT =====")

for message in authorized_cancel_result["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== DATABASE AFTER CANCELLATION =====")
show_booking_database()

===== AUTHORIZED CANCELLATION RESULT =====

 HumanMessage : I want to cancel my booking ID 1. Please proceed.

 AIMessage : 

 ToolMessage : Booking 1 cancelled successfully.

 AIMessage : Your booking with ID 1 has been successfully cancelled. If you need anything else—such as rescheduling, booking a new test, or any other assistance—just let me know!

===== DATABASE AFTER CANCELLATION =====
===== CURRENT BOOKINGS =====
[(1, 'user_001', 1, 7, 'cancelled')]

===== SLOT COUNTS =====
(1, 1, 0)
(2, 1, 0)
(3, 1, 0)
(4, 1, 0)
(5, 1, 0)
(6, 1, 0)
(7, 1, 0)
(8, 1, 0)
(9, 1, 0)
(10, 1, 1)
(11, 1, 1)
(12, 1, 1)
(13, 1, 0)
(14, 1, 0)
(15, 1, 0)
(16, 1, 0)
(17, 1, 0)
(18, 1, 0)
(19, 1, 0)
(20, 1, 0)
(21, 1, 0)
(22, 1, 0)
(23, 1, 0)
(24, 1, 0)


In [44]:

conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()
cursor.execute("""
    SELECT
        t.test_name,
        s.slot_id,
        s.slot_datetime,
        s.capacity,
        s.booked_count
    FROM tests t
    JOIN slots s ON t.test_id = s.test_id
    WHERE t.test_name = 'Hemoglobin (HB)'
    ORDER BY s.slot_id
""")

for row in cursor.fetchall():
    print(row)

conn.close()

('Hemoglobin (HB)', 13, '2026-12-01 09:00:00', 1, 0)
('Hemoglobin (HB)', 14, '2026-12-01 10:00:00', 1, 0)
('Hemoglobin (HB)', 15, '2026-12-01 11:00:00', 1, 0)


In [45]:
second_booking_thread = "second-booking-user-002"

second_booking_request = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="I want to book Hemoglobin (HB) for slot 13."
            )
        ]
    },
    config={"configurable": {"thread_id": second_booking_thread}},
    context=UserContext(user_id="user_002")
)

print("===== SECOND BOOKING REQUEST =====")

for message in second_booking_request["messages"]:
    print("\n", type(message).__name__, ":", message.content)

===== SECOND BOOKING REQUEST =====

 HumanMessage : I want to book Hemoglobin (HB) for slot 13.

 AIMessage : 

 ToolMessage : AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Category: Bioc

In [46]:
second_booking_confirmation = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Yes, I confirm. Please book it.")
        ]
    },
    config={"configurable": {"thread_id": second_booking_thread}},
    context=UserContext(user_id="user_002")
)

print("===== SECOND BOOKING APPROVAL REQUEST =====")

for message in second_booking_confirmation["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== APPROVAL INTERRUPT =====")
print(second_booking_confirmation.get("__interrupt__", "No interrupt found"))

===== SECOND BOOKING APPROVAL REQUEST =====

 HumanMessage : I want to book Hemoglobin (HB) for slot 13.

 AIMessage : 

 ToolMessage : AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Categ

In [47]:
from langgraph.types import Command

second_booking_result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config={"configurable": {"thread_id": second_booking_thread}},
    context=UserContext(user_id="user_002")
)

print("===== SECOND BOOKING RESULT =====")

for message in second_booking_result["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== DATABASE AFTER BOOKING =====")
show_booking_database()

===== SECOND BOOKING RESULT =====

 HumanMessage : I want to book Hemoglobin (HB) for slot 13.

 AIMessage : 

 ToolMessage : AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Category: Bioch

In [48]:
conflict_thread = "conflict-test-user-001"

conflict_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="I want to book Electrocardiogram (ECG) for slot 10."
            )
        ]
    },
    config={"configurable": {"thread_id": conflict_thread}},
    context=UserContext(user_id="user_001")
)

print("===== CONFLICT TEST =====")

for message in conflict_response["messages"]:
    print("\n", type(message).__name__, ":", message.content)

print("\n===== DATABASE AFTER CONFLICT TEST =====")
show_booking_database()

===== CONFLICT TEST =====

 HumanMessage : I want to book Electrocardiogram (ECG) for slot 10.

 AIMessage : 

 ToolMessage : AVAILABLE DIAGNOSTIC TESTS:
ID: 1 | Test: Complete Blood Count (CBC) | Category: Hematology | Description: Measures blood cells and hemoglobin. | Price: Rs. 450.00
ID: 2 | Test: Hemoglobin (HB) | Category: Hematology | Description: Measures hemoglobin levels in blood. | Price: Rs. 200.00
ID: 3 | Test: Blood Group Test | Category: Hematology | Description: Identifies ABO and Rh blood groups. | Price: Rs. 250.00
ID: 4 | Test: Blood Glucose (Fasting) | Category: Biochemistry | Description: Measures fasting blood glucose. | Price: Rs. 150.00
ID: 5 | Test: Lipid Profile | Category: Biochemistry | Description: Measures cholesterol and triglycerides. | Price: Rs. 800.00
ID: 6 | Test: Liver Function Test (LFT) | Category: Biochemistry | Description: Measures markers related to liver function. | Price: Rs. 700.00
ID: 7 | Test: Kidney Function Test (KFT) | Category: Bioch

In [49]:
health_info_thread = "health-info-test"

health_info_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="What does a Complete Blood Count (CBC) test measure?"
            )
        ]
    },
    config={"configurable": {"thread_id": health_info_thread}},
    context=UserContext(user_id="user_001")
)

print("===== GENERAL HEALTH INFORMATION TEST =====")

for message in health_info_response["messages"]:
    print("\n", type(message).__name__, ":", message.content)

===== GENERAL HEALTH INFORMATION TEST =====

 HumanMessage : What does a Complete Blood Count (CBC) test measure?

 AIMessage : 

 ToolMessage : {'query': 'Complete Blood Count CBC test measures', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.ucsfhealth.org/care/medical-tests/cbc-blood-test', 'title': 'CBC blood test | UCSF Health', 'content': '# CBC blood test\n\n#### Definition\n\nA complete blood count (CBC) test measures the following:\n\n- The number of white blood cells (WBC count)\n- The number of red blood cells (RBC count)\n- The number of platelets\n- The total amount of hemoglobinin the blood\n- The fraction of the blood composed of red blood cells (hematocrit)\n\nThe CBC test also provides information about the following measurements: [...] - Mean red blood cell volume (MCV)\n- Mean hemoglobinamount per red blood cell (MCH)\n- The mean amount of hemoglobin relative to the size of the cell (hemoglobin concentration) per red blood

In [50]:
symptom_thread = "symptom-test-user-001"

symptom_response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="I have fever and fatigue. Which lab test should I take?"
            )
        ]
    },
    config={"configurable": {"thread_id": symptom_thread}},
    context=UserContext(user_id="user_001")
)

print("===== SYMPTOM HANDLING TEST =====")

for message in symptom_response["messages"]:
    print("\n", type(message).__name__, ":", message.content)

===== SYMPTOM HANDLING TEST =====

 HumanMessage : I have fever and fatigue. Which lab test should I take?

 AIMessage : I’m sorry you’re feeling unwell. Fever and fatigue can be caused by many different conditions, and selecting the appropriate laboratory test depends on a thorough clinical evaluation.

I recommend that you contact a qualified healthcare professional (your primary‑care doctor, a nurse practitioner, or another medical provider) who can assess your symptoms, medical history, and any other relevant factors. They’ll be able to determine whether any lab tests are needed and, if so, which specific test(s) would be most appropriate for you. 

If you later receive a specific test recommendation from your doctor and would like help checking availability or booking an appointment, just let me know—I’ll be happy to assist!


In [51]:
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.types import Command

In [52]:

while True:
    print("\n===== LAB TEST BOOKING ASSISTANT =====")

    user_id = input("Enter user ID (user_001 or user_002): ").strip()

    if user_id.lower() == "exit":
        print("Assistant closed.")
        break

    if user_id not in ["user_001", "user_002"]:
        print("Invalid user ID. Please try again.")
        continue

    user_input = input("You: ").strip()

    if user_input.lower() == "exit":
        print("Assistant closed.")
        break

    if not user_input:
        continue

    # Redact email before sending the message to the agent
    safe_message = redact_email(user_input)

    thread_id = f"interactive-{user_id}"

    response = agent.invoke(
        {"messages": [HumanMessage(content=safe_message)]},
        config={"configurable": {"thread_id": thread_id}},
        context=UserContext(user_id=user_id)
    )

    # Display tool calls
    for message in response["messages"]:
        if isinstance(message, AIMessage) and message.tool_calls:
            for call in message.tool_calls:
                print("\nTOOL CALL:", call["name"])
                print("Arguments:", call["args"])

    # Handle approval requests
    while response.get("__interrupt__"):
        interrupt_data = response["__interrupt__"][0].value

        print("\n===== APPROVAL REQUIRED =====")
        for action in interrupt_data["action_requests"]:
            print("Action:", action["name"])
            print("Arguments:", action["args"])

        decision = input("Approve or reject? ").strip().lower()

        if decision == "approve":
            approval_decision = {"type": "approve"}
        else:
            approval_decision = {
                "type": "reject",
                "message": "User rejected this action."
            }

        response = agent.invoke(
            Command(resume={"decisions": [approval_decision]}),
            config={"configurable": {"thread_id": thread_id}},
            context=UserContext(user_id=user_id)
        )

        for message in response["messages"]:
            if isinstance(message, ToolMessage):
                print("\nTOOL RESULT:", message.content)

    # Display final answer
    for message in reversed(response["messages"]):
        if isinstance(message, AIMessage) and message.content:
            print("\nAssistant:", message.content)
            break



===== LAB TEST BOOKING ASSISTANT =====


StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

In [ ]:

import pandas as pd

evaluation_results = [
    {
        "Scenario": "1. Availability check",
        "Expected Tools": "list_tests, check_slots",
        "Actual Tools": "list_tests, check_slots",
        "Expected DB Change": "No change",
        "Actual DB Change": "No change",
        "Result": "PASS"
    },
    {
        "Scenario": "2. Successful booking",
        "Expected Tools": "list_tests, check_slots, book_test",
        "Actual Tools": "list_tests, check_slots, book_test",
        "Expected DB Change": "Create booking; increase slot count",
        "Actual DB Change": "Booking 1 created; slot count increased",
        "Result": "PASS"
    },
    {
        "Scenario": "3. Second user booking",
        "Expected Tools": "list_tests, check_slots, book_test",
        "Actual Tools": "list_tests, check_slots, book_test",
        "Expected DB Change": "Create second booking",
        "Actual DB Change": "Booking 2 created for user_002",
        "Result": "PASS"
    },
    {
        "Scenario": "4. Booking rejection",
        "Expected Tools": "book_test (rejected)",
        "Actual Tools": "book_test (rejected)",
        "Expected DB Change": "No change",
        "Actual DB Change": "No booking created",
        "Result": "PASS"
    },
    {
        "Scenario": "5. Slot conflict",
        "Expected Tools": "list_tests, check_slots",
        "Actual Tools": "list_tests, check_slots",
        "Expected DB Change": "No change",
        "Actual DB Change": "No booking created",
        "Result": "PASS"
    },
    {
        "Scenario": "6. Booking cancellation",
        "Expected Tools": "cancel_booking",
        "Actual Tools": "cancel_booking",
        "Expected DB Change": "Cancel booking; decrease slot count",
        "Actual DB Change": "Booking 1 cancelled; slot count decreased",
        "Result": "PASS"
    },
    {
        "Scenario": "7. General health information",
        "Expected Tools": "search_health_information",
        "Actual Tools": "search_health_information",
        "Expected DB Change": "No change",
        "Actual DB Change": "No change",
        "Result": "PASS"
    },
    {
        "Scenario": "8. Symptom question",
        "Expected Tools": "No tools",
        "Actual Tools": "No tools",
        "Expected DB Change": "No change",
        "Actual DB Change": "No change",
        "Result": "PASS"
    }
]

evaluation_df = pd.DataFrame(evaluation_results)

print("===== FINAL EVALUATION RESULTS =====")
display(evaluation_df)

print("\nTotal Scenarios:", len(evaluation_df))
print("Passed:", (evaluation_df["Result"] == "PASS").sum())
print("Failed:", (evaluation_df["Result"] == "FAIL").sum())
